# Homework 3: Information Theory and Geometry

## Part 1: The Distance Engine (K-NN)
You must build a K-Nearest Neighbors classifier from scratch using `numpy`. 
Since K-NN has no training phase, `fit` just stores the data. The heavy lifting happens in `predict`.

**Your Task:**
Complete the `CustomKNN` class.
1. In `predict`, calculate the Euclidean distance between the new point and ALL training points.
2. Sort the distances to find the indices of the `k` closest neighbors.
3. Return the majority class label among those neighbors.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

class CustomKNN:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        # K-NN is lazy. Just store the data!
        self.X_train = X
        self.y_train = y
        
    def predict(self, X):
        predictions = []
        for x in X:
            # TODO: Calculate Euclidean distances from 'x' to all self.X_train
            
            # TODO: Get the indices of the k smallest distances
            
            # TODO: Get the labels of those k neighbors
            
            # TODO: Find the most common label and append to predictions
            pass
            
        return np.array(predictions)

# --- Test your KNN ---
X_hw, y_hw = make_classification(n_samples=300, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_hw, y_hw, test_size=0.2, random_state=42)

# TODO: Train CustomKNN, predict, and print accuracy score

## Part 2: The Recursive Decision Tree Engine
A Decision Tree is not just a single split; it is a recursive algorithm that builds a tree of `Node` objects. 

To prevent the tree from growing infinitely and overfitting, we use **Pre-Pruning** hyperparameters: `max_depth` and `min_samples_split`.

**Your Task:**
Complete the `CustomDecisionTree` class. We have provided the recursive `_build_tree` structure. You must implement:
1. The **Stopping Criteria** (Pre-Pruning) at the top of the build function.
2. The **Shannon Entropy** calculation.
3. The **Information Gain** calculation.
4. The **Best Split** finder.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

class Node:
    """A helper class to store the structure of the tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value # Only leaf nodes will have a value (the predicted class)
        
    def is_leaf_node(self):
        return self.value is not None

class CustomDecisionTree:
    def __init__(self, max_depth=100, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        
    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        
    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # --- 1. STOPPING CRITERIA (PRE-PRUNING) ---
        # TODO: Check if we should stop growing and return a leaf node.
        # Stop if:
        # A) depth is >= self.max_depth
        # B) n_labels == 1 (Node is 100% pure)
        # C) n_samples < self.min_samples_split
        if False: # Replace False with your combined conditions
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
            
        # --- 2. FIND THE BEST SPLIT ---
        best_feat, best_thresh = self._best_split(X, y, n_features)
        
        # If no split improves information gain, it returns None. Make it a leaf.
        if best_feat is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
            
        # --- 3. RECURSIVELY BUILD CHILDREN ---
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left_child = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(best_feat, best_thresh, left_child, right_child)
        
    def _best_split(self, X, y, n_features):
        best_gain = -1
        split_idx, split_thresh = None, None
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thr in thresholds:
                # TODO: Calculate Information Gain for this specific feature and threshold
                gain = 0 # Replace with your calculation using self._information_gain()
                
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thr
                    
        return split_idx, split_thresh

    def _information_gain(self, y, X_column, threshold):
        # TODO: Implement Information Gain
        # 1. Calculate parent entropy
        # 2. Generate split (get left and right indices)
        # 3. Calculate weighted average entropy of children
        # 4. Return Parent Entropy - Child Entropy
        pass

    def _entropy(self, y):
        # TODO: Calculate Shannon Entropy
        pass

    def _split(self, X_column, split_thresh):
        """Helper to return row indices for left and right splits."""
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    # --- PREDICTION LOGIC (PROVIDED) ---
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# --- Test the Engine ---
X_hw, y_hw = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_hw, y_hw, test_size=0.2, random_state=42)

# TODO: Train CustomDecisionTree with max_depth=3, predict, and print accuracy

# TODO: Train sklearn's DecisionTreeClassifier with max_depth=3, predict, and print accuracy
# (They should be very close, if not identical!)